# Scale Down Fabric Capacity to F64

This Fabric notebook scales a Microsoft Fabric capacity to **F64**.

## What it does
1. Gets the current capacity SKU (idempotent guard).
2. If already on target SKU, exits cleanly.
3. Otherwise PATCHes the capacity SKU via Azure Resource Manager.
4. Polls the async operation until completion.
5. Re-reads the capacity to confirm the new SKU.

## Prereqs
- Run in a Microsoft Fabric notebook runtime (for `notebookutils`).
- Azure permissions on the capacity resource (Owner/Contributor or a custom role with `Microsoft.Fabric/capacities/write`).

Fill in the parameters in the next cell.


In [1]:
# === USER INPUTS ===
SUBSCRIPTION_ID = "your-subscription-id"
RESOURCE_GROUP  = "your-resource-group"
CAPACITY_NAME   = "your-capacity-name"   # Azure resource name

# Optional: provide the full ARM resource ID directly
CAPACITY_RESOURCE_ID = ""  # e.g. /subscriptions/.../resourceGroups/.../providers/Microsoft.Fabric/capacities/<name>

# Target SKU for THIS notebook
#TARGET_SKU = "F128"   # ScaleUp notebook
 TARGET_SKU = "F64"  # ScaleDown noteboo

# Recommended API version (update if your org standardizes on a different one)
API_VERSION = "2023-11-01"

# Safety: only allow scaling on weekdays
WEEKDAYS_ONLY = True

if not CAPACITY_RESOURCE_ID:
    CAPACITY_RESOURCE_ID = (
        f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.Fabric/capacities/{CAPACITY_NAME}"
    )

CAPACITY_RESOURCE_ID


StatementMeta(, 577c3d52-1744-4277-9f99-a5e6c3e67272, 3, Finished, Available, Finished)

IndentationError: unexpected indent (539204208.py, line 11)

##### **Assign service principal credential information**
Update the **keyVaultEndpoint** to the Azure Key Vault url and the secret name values if the service principal credentials are stored there.\
These credential information can be hard coded for testing purposes and to get started.

In [2]:
from notebookutils import mssparkutils

keyVaultEndpoint = 'your-key-vault'

tenantId = mssparkutils.credentials.getSecret(keyVaultEndpoint, 'sp-tenant-id')
clientId = mssparkutils.credentials.getSecret(keyVaultEndpoint, 'sp-client-id')
secret = mssparkutils.credentials.getSecret(keyVaultEndpoint, 'sp-client-secret')

StatementMeta(, 217f5517-49e9-44c2-bbce-a895c1f576aa, 4, Finished, Available, Finished)

##### **Acquire Tokens and create the API headers**
We need to acquire two tokens:
- PBI audience so that we're able to use the PBI/Fabric APIs.
- Azure Management audience to scale the capacity within Azure.

In [3]:
from azure.identity import ClientSecretCredential

api_pbi = "https://analysis.windows.net/powerbi/api/.default"
api_arm = "https://management.azure.com/.default"  # <-- use this for ARM

auth = ClientSecretCredential(tenant_id=tenantId, client_id=clientId, client_secret=secret)

header_pbi = {"Authorization": f"Bearer {auth.get_token(api_pbi).token}", "Content-type": "application/json"}
header_arm = {"Authorization": f"Bearer {auth.get_token(api_arm).token}", "Content-type": "application/json"}

StatementMeta(, 217f5517-49e9-44c2-bbce-a895c1f576aa, 5, Finished, Available, Finished)

##### **Verify the Current SKU**


In [4]:
import requests

response = requests.get("https://api.fabric.microsoft.com/v1/capacities", headers=header_pbi)

currentSku = [capacity.get('sku') for capacity in response.json().get('value') if capacity.get('displayName') == CAPACITY_NAME][0]
print(f'{currentSku = }')

StatementMeta(, 217f5517-49e9-44c2-bbce-a895c1f576aa, 6, Finished, Available, Finished)

currentSku = 'F128'


##### **Perform the scaling operation within Azure**
If the current SKU of the capacity is different than the target SKU, then perform the scaling operation.

In [2]:
import requests, json

# Validation to check if the sku to scale to is different than the current sku
if TARGET_SKU != currentSku:
    print(f'\nScaling from {currentSku} to {TARGET_SKU}')

    response = requests.get(f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/providers/Microsoft.Fabric/capacities?api-version=2022-07-01-preview', headers=header_arm)
    responseList = response.json().get('value')
    resourceGroupName = RESOURCE_GROUP

    url = f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{resourceGroupName}/providers/Microsoft.Fabric/capacities/{CAPACITY_NAME}?api-version=2022-07-01-preview'
    body = {"sku": {"name": f"{TARGET_SKU}", "tier": "Fabric"}}
            
    response = requests.patch(url, headers=header_arm, data=json.dumps(body))
    print(response, response.text)

else:
    print(f'The current SKU is already {TARGET_SKU}')

StatementMeta(, 577c3d52-1744-4277-9f99-a5e6c3e67272, 4, Finished, Available, Finished)

NameError: name 'TARGET_SKU' is not defined